# Runtime

Copy page

​
Overview

LangChain’s create_agent runs on LangGraph’s runtime under the hood.

LangGraph exposes a Runtime object with the following information:

Context: static information like user id, db connections, or other dependencies for an agent invocation

Store: a BaseStore instance used for long-term memory

Stream writer: an object used for streaming information via the "custom" stream mode

Execution info: identity and retry information for the current execution (thread ID, run ID, attempt number)

Server info: server-specific metadata when running on LangGraph Server (assistant ID, graph ID, authenticated user)

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

In [ ]:
import pprint
from langchain.tools import tool

from langchain.chat_models import init_chat_model
from rich import print as rprint

In [ ]:
model_free = init_chat_model("openai/gpt-oss-20b",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=10000, temperature=0.0)

model_basic = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)

model_medium = init_chat_model("openai/gpt-5.6-luna",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

model_advanced = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

model_safety = init_chat_model("nvidia/nemotron-3.5-content-safety:free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)



In [ ]:
from dataclasses import dataclass

from langchain.agents import create_agent


@dataclass
class Context:
    user_name: str

agent = create_agent(
    model="gpt-5-nano",
    tools=[...],
    context_schema=Context  
)


In [ ]:

agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    context=Context(user_name="John Smith")
)

## Inside tools

You can access the runtime information inside tools to:

Access the context

Read or write long-term memory

Write to the custom stream (ex, tool progress / updates)

Use the ToolRuntime parameter to access the Runtime object inside a tool.

In [ ]:

from langchain.tools import tool, ToolRuntime  

@tool
def fetch_user_email_preferences(runtime: ToolRuntime[Context]) -> str:
    """Fetch the user's email preferences from the store."""
    user_id = runtime.context.user_id  

    preferences: str = "The user prefers you to write a brief and polite email."
    if runtime.store:
        if memory := runtime.store.get(("users",), user_id):
            preferences = memory.value["preferences"]

    return preferences

In [ ]:

agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    context=Context(user_name="John Smith")
)